# 📘 부록: OKF 비서 만들기 1. 묻고 답하기

OKF 번들을 읽고 질문에 답하는 비서를 만듭니다. 먼저 Codex, Claude Code 같은 코딩 에이전트에 규칙 파일 하나만 주고 번들로 답하게 해 봅니다. 이어서 같은 비서를 코드로 직접 만듭니다. Jev가 어느 번들인지 고르고, LangChain 에이전트가 목차부터 읽어 필요한 문서만 열고, 답에는 출처 개념 ID와 신뢰 단계를 붙입니다. 번들 고르기 기준도 각 번들의 목차에서 읽으므로, 부록 01의 예시 번들이 아닌 다른 위키에도 같은 코드를 씁니다.

**실행 환경:** CPU만 씁니다. 1절은 Codex CLI나 Claude Code가 설치된 터미널에서 따라 하고, 2절부터는 `.env`의 `OPENAI_API_KEY`와 `TYPESAFE_API_KEY`가 필요합니다. 질문 하나에 Jev 한 번, GPT 두세 번을 부릅니다. 공식 문서: [OKF SPEC v0.2](https://github.com/GoogleCloudPlatform/open-knowledge-format/blob/main/SPEC.md)

**학습 목표**

1. **규칙 파일(`AGENTS.md`)로 Codex와 Claude Code가 번들을 읽고 출처를 붙여 답하게 합니다.**
2. **Jev의 Choice로 질문에 맞는 번들을 고릅니다.**
3. **목차 읽기, 문서 읽기를 LangChain 도구로 만들고, 목차부터 읽고 출처를 붙여 답하는 비서 에이전트를 만듭니다.**
4. **초안, 만료, 검증 전 문서를 답에서 알리게 합니다.**

## 1. 코딩 에이전트에서 바로 쓰기

### 1-1. 규칙 파일 하나면 됩니다

Codex와 Claude Code는 폴더의 파일을 스스로 열어 읽습니다. 그래서 '어떻게 읽고 어떻게 답할지'를 적은 규칙 파일만 두면 바로 위키 비서가 됩니다. 이 규칙 파일이 부록 01에서 본 LLM 위키의 스키마 층입니다.

규칙 파일은 위키가 아니므로 번들 밖, `data/` 바로 아래에 둡니다. 그러면 `data/`의 세 갈래가 LLM 위키의 세 층과 그대로 맞습니다.

| 위치 | 층 | 질문에 답할 때 |
|---|---|---|
| `data/raw/` | 원본 자료 | 읽지 않습니다. 위키에 넣는 일은 부록 03에서 합니다 |
| `data/bundles/` | 위키 | 목차부터 읽고 필요한 문서만 열어 답합니다 |
| `data/AGENTS.md` | 스키마 | 위의 두 가지를 어떻게 할지 정합니다 |

규칙 파일을 번들 폴더(`studio/`, `game_data/`) 안에 두면 개념 문서처럼 읽히고, frontmatter가 없어 적합성 조건(부록 01 6절)도 어깁니다.

`data/AGENTS.md`

~~~markdown
# OKF 위키 비서 규칙

이 폴더는 LLM 위키의 세 층으로 나뉜다. bundles/ 아래의 폴더 하나가 OKF 번들(위키) 하나이고, studio는 업무 위키, game_data는 데이터 위키다. raw/는 원본 자료다. 위키 문서의 근거가 된 원문(회의 메모 등)과 비서가 새로 받은 자료가 들어 있고, 위키에 넣은 뒤에도 원문은 그대로 둔다. 이 파일은 위키를 읽고 쓰는 규칙이다.

## 위키 형식(OKF)

- 번들 안의 .md 파일 하나가 개념 문서 하나다. 번들 안 경로에서 .md를 뺀 것이 개념 ID다(예: playbooks/server_incident).
- index.md는 목차다. 번들 맨 위 index.md에는 폴더가, 폴더 안 index.md에는 그 폴더의 문서가 한 줄씩 적혀 있다. log.md는 번들의 변경 기록이다. 둘은 개념 문서가 아니다.
- 개념 문서는 맨 위 --- 두 줄 사이의 frontmatter와 그 아래 본문으로 나뉜다. frontmatter 키의 뜻: type은 문서 종류, description은 한 줄 설명, status는 draft(초안), stable(기본값), deprecated(기록용), stale_after는 이 시각이 지나면 내용이 낡았을 수 있다는 표시, sources는 근거 자료, generated는 누가 썼는지, verified는 누가 내용을 확인했는지다.
- 본문의 링크 [이름](/폴더/파일.md)는 번들 맨 위 기준 경로다.

## 질문에 답할 때

1. 질문에 맞는 번들의 맨 위 index.md(bundles/<번들>/index.md)부터 읽고, 필요한 폴더의 index.md를 읽은 뒤 필요한 개념 문서만 연다. ls, find, rg로 폴더 전체를 먼저 훑지 않는다.
2. 답에 쓰는 사실은 bundles/에서 읽은 개념 문서에서만 가져온다. raw/의 원본 자료는 답할 때 읽지 않는다. 문서에 없는 값이나 계산식은 만들지 말고 모른다고 답한다.
3. 읽은 문서의 frontmatter에서 status가 draft나 deprecated이거나 stale_after가 오늘보다 앞이면, 답의 첫 줄을 '주의: '로 시작해 그 사실을 적는다(stale_after가 지났으면 내용이 낡았을 수 있다고 적는다).
4. 답은 한국어로 짧게 쓰고, 마지막 줄에 출처를 적는다.
   - 형식: `출처: 번들 이름/개념 ID(신뢰 단계)`. 쓴 문서가 여럿이면 쉼표로 잇는다.
   - 번들 이름과 개념 ID의 하위 폴더를 모두 적는다. 앞에 bundles/를 붙이지 않고, 끝에 .md를 붙이지 않는다.
   - 맞는 예: `출처: studio/playbooks/server_incident(검증 전), game_data/metrics/dau(검증 전)`
   - 틀린 예: `playbooks/server_incident`(번들 이름 빠짐), `game_data/dau`(하위 폴더 빠짐), `bundles/studio/playbooks/server_incident`(bundles/ 붙음), `studio/playbooks/server_incident.md`(.md 붙음)
   - 답에 쓴 문서가 없으면 `출처: 없음`이라고 적는다.
   - 신뢰 단계는 verified 키만 보고 정한다. verified가 없으면 검증 전, verified에 human: 기록이 있으면 사람 검토, 그 밖의 기록만 있으면 기계 확인이다. generated(누가 썼는지)는 신뢰 단계와 상관없다.

## 문서를 고칠 때

- 사람이 고치라고 지시할 때만 고친다. 질문에 답하다가 문서를 고치지 않는다.
- 고칠 수 있는 것은 bundles/ 안의 파일(개념 문서, index.md, log.md)뿐이다.
- raw/의 원본 자료는 읽기만 하고 고치거나 지우거나 옮기지 않는다. 원문을 위키 문서에 그대로 붙이지 않고, 정리한 사실만 적은 뒤 sources로 원문을 가리킨다.
- 규칙 파일(AGENTS.md)은 사람만 고친다.
- 문서를 고치면 그 번들 log.md의 오늘 날짜 제목 아래에 무엇을 고쳤는지 한 줄 적는다.
~~~

- **Codex**는 작업 폴더의 `AGENTS.md`를 읽습니다.
- **Claude Code**(2.1.277 이상)도 작업 폴더와 그 위 폴더에 `CLAUDE.md`가 없으면 `AGENTS.md`를 읽습니다. 그래서 규칙은 `AGENTS.md` 한 파일에만 적고 두 도구가 같이 씁니다.
- **두 부분:** `AGENTS.md`는 위키가 어떤 형식인지(위키 형식)와 어떻게 찾고 답할지(질문에 답할 때)를 적습니다. 형식을 적어 두지 않으면 에이전트가 frontmatter 키의 뜻을 짐작하다 틀립니다(1-3의 `generated`와 `verified` 혼동). OKF SPEC 전체가 아니라 답하는 데 필요한 형식만 적습니다. 규칙 파일은 질문할 때마다 통째로 읽히기 때문입니다.

**규칙 파일을 찾는 위치**

두 도구는 실행한 폴더(작업 폴더)를 기준으로 규칙 파일을 찾습니다.

- **Codex:** 프로젝트 맨 위(git 저장소 최상위)부터 작업 폴더까지 내려오며 폴더마다 `AGENTS.md`를 읽습니다. 작업 폴더보다 아래 폴더의 것은 읽지 않습니다.
- **Claude Code:** 작업 폴더와 그 위 폴더들의 `AGENTS.md`를 시작할 때 읽습니다(그 폴더들에 `CLAUDE.md`가 있으면 `CLAUDE.md`를 대신 읽음). 아래 폴더의 것은 그 폴더의 파일을 열 때에야 읽습니다.
- **이 부록:** 그래서 1-2에서는 `cd data`로 `data/`를 작업 폴더로 만든 뒤 실행합니다. 두 도구 모두 시작할 때 `data/`의 규칙 파일을 읽습니다.
- **맨 위 폴더에서 실행할 때:** Codex는 `codex -C data`로 작업 폴더를 지정하고, Claude Code는 맨 위 `CLAUDE.md`에 `@data/AGENTS.md` 한 줄을 넣어 불러옵니다.

### 1-2. 직접 물어보기

터미널에서 코딩 에이전트를 실행하고, 대화 창에 질문을 쳐서 답을 확인합니다. 아래에서 **⌨️ 입력**은 치는 내용이고, **💬 답**은 그 아래에 나오는 비서의 답입니다.

**준비:** Claude Code나 Codex CLI 가운데 하나가 설치되어 로그인되어 있어야 합니다. 터미널에서 `claude --version`이나 `codex --version`을 쳐서 버전이 나오면 됩니다. Claude Code는 2.1.277 이상이어야 `AGENTS.md`를 읽습니다.

**1단계. `data` 폴더에서 도구를 실행합니다.** 규칙 파일이 있는 폴더에서 실행해야 에이전트가 규칙을 읽습니다. 이 부록 폴더에서 `data`로 옮긴 뒤 쓰는 도구 하나를 실행합니다. 폴더를 믿을지 묻는 창이 나오면 허용합니다.

⌨️ 입력 (터미널, Claude Code)

```bash
cd data
claude
```

⌨️ 입력 (터미널, Codex)

```bash
cd data
codex
```

**2단계. 대화 창에 질문을 칩니다.**

⌨️ 입력 (대화 창)

~~~text
서버 장애가 나면 누구에게 먼저 연락해야 해?
~~~

**3단계. 답을 읽습니다.** 아래는 Claude Code가 낸 답의 예입니다. 문장은 도구와 실행마다 조금씩 다르지만, 아래 세 부분이 있어야 합니다.

💬 답

```text
주의: 이 문서는 stale_after(2026-10-01)가 지나서 내용이 낡았을 수 있습니다.

서버 장애가 나면 1차 담당인 박서연에게 먼저 전화합니다. 전화번호는 사내 연락망 7번입니다. 그 전에 상태 확인 명령으로 장애인지 확인하고, 상태 페이지에 '점검 중'을 올립니다. 30분 안에 복구되지 않으면 윤하나에게 알립니다.

출처: studio/playbooks/server_incident(검증 전)
```

| 답의 부분 | 무엇을 말하나 | 규칙 |
|---|---|---|
| 첫 줄 `주의:` | 서버 장애 대응 문서의 `stale_after`가 오늘보다 앞이라 낡았을 수 있음 | 3번 |
| 가운데 | 그 문서에 적힌 순서와 연락처만 씀(명령어 자체는 묻지 않아 생략) | 2번 |
| 마지막 줄 `출처:` | 쓴 문서의 번들 이름과 개념 ID(`studio` + `playbooks/server_incident`), 신뢰 단계(`verified`가 없어 검증 전) | 4번 |

**4단계. 질문을 바꿔 봅니다.** 같은 대화 창에 아래 질문을 이어서 칩니다. 규칙대로라면 답의 첫 줄은 표와 같고, 출처 줄에는 표의 문서가 들어 있어야 합니다. 에이전트가 링크를 따라 다른 문서도 읽으면 출처에 그 문서가 더 붙을 수 있습니다. 다르게 나오면 1-3처럼 규칙 파일을 고칠 곳입니다.

| 질문 | 답의 첫 줄 | 출처 줄에 들어 있을 문서 |
|---|---|---|
| 7일 잔존율은 어떻게 계산하고, 최근 값은 얼마야? | 주의 없음 | `game_data/metrics/d7_retention(사람 검토)` |
| 게임 로그는 언제부터 언제까지 있어? | 주의 없음 | `game_data/datasets/game_logs(기계 확인)` |
| 결제 전환율은 어떻게 계산하고, 최근 값은 얼마야? | `주의:` 초안 문서. 계산 방법과 최근 값은 문서에 없어 모른다고 답함 | `game_data/metrics/paying_conversion(검증 전)` |
| 추석 이벤트는 지금 진행 중이야? | `주의:` deprecated(기록용) 문서 | `studio/projects/chuseok_event(검증 전)` |
| 회사 와이파이 비밀번호 알려 줘 | 주의 없음(본문은 위키에 없어 모른다고 답함) | `출처: 없음` |


### 1-3. 규칙 파일은 쓰는 LLM에 맞게 고칩니다

`AGENTS.md`는 한 번 쓰고 끝내는 문서가 아닙니다. 쓰는 모델이 실제로 무엇을 틀리는지 돌려 보고, 틀린 곳마다 규칙을 한 줄씩 더합니다. 위 규칙도 이렇게 고친 결과입니다.

| 모델이 틀린 것 | 더한 규칙 |
|---|---|
| 목차보다 먼저 폴더 전체를 훑음(`ls -R`) | 1번: 목차부터 읽고, 폴더를 먼저 훑지 않는다 |
| 문서에 없는 계산식을 지어냄 | 2번: 문서에 없는 값은 만들지 않는다 |
| 출처에서 하위 폴더를 빼먹음 | 4번: 맞는 예와 틀린 예를 적는다 |
| `generated: human:...`을 보고 '사람 검토'로 적음 | 4번: 신뢰 단계는 `verified`만 본다 |

모델이나 도구 버전이 바뀌면 틀리는 곳도 바뀌므로, 바꾼 뒤에는 같은 질문으로 다시 돌려 봅니다.

### 1-4. 코드로 직접 만드는 이유

코딩 에이전트는 개발자 PC의 터미널에서 사람이 직접 쓸 때 편합니다. 사내 메신저나 웹 서비스에 비서를 붙이려면 코드로 만들어야 합니다. 직접 만들면 1-3에서 본 것처럼 모델이 틀릴 수 있는 일을 코드에 맡길 수 있습니다.

- **번들 고르기:** 모델이 짐작하지 않고 Jev가 고릅니다(3절).
- **상태, 신뢰 단계, 만료:** 모델이 frontmatter를 해석하지 않고 코드가 계산해 문서 앞에 붙입니다(4절).
- **목차부터 읽기:** 맨 위 목차는 코드가 메시지에 넣어 줍니다(5절).

2절부터 이 비서를 LangChain으로 만듭니다.

## 2. 실습 준비

<img src="images/06_secretary_query.png" width="1050" alt="질문이 오면 Jev가 두 번들 가운데 하나를 고르고, 비서가 목차부터 읽어 필요한 개념 문서를 신뢰 단계와 함께 읽은 뒤 GPT가 주의와 출처를 붙여 답합니다.">

### 2-1. 경로, 키, 모델

번들 문서를 frontmatter와 본문으로 나눠 읽는 함수(`read_concept`)와 신뢰 단계(`trust_tier`), 만료(`is_stale`)를 판정하는 함수는 `okf_utils.py`에 있습니다. 부록 01 5절의 판정 기준을 코드로 옮긴 것입니다.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

material_dir = Path(".")  # 이 부록 폴더
bundles_dir = material_dir / "data" / "bundles"

# 맨 위 index.md가 있는 폴더가 번들입니다(번들을 더해도 코드는 그대로)
BUNDLES = []
for path in sorted(bundles_dir.iterdir()):
    if (path / "index.md").is_file():
        BUNDLES.append(path.name)

# 샘플 번들의 '오늘'(시나리오 날짜). 만료 판정의 기준입니다
NOW = datetime(2026, 11, 2, 9, 0, tzinfo=timezone(timedelta(hours=9)))

from okf_utils import is_stale, read_concept, trust_tier

In [ ]:
import os

from dotenv import load_dotenv

# 부록 폴더, day72 폴더, 실습자료 폴더의 .env를 차례로 읽습니다(먼저 읽은 값이 남습니다)
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir.resolve().parent.parent / ".env")
print("OpenAI 키:", os.getenv("OPENAI_API_KEY") is not None, "| Jev 키:", os.getenv("TYPESAFE_API_KEY") is not None)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_typesafe import TypeSafeClassifier

model_name = "gpt-4.1-mini"
llm = ChatOpenAI(model=model_name, use_responses_api=True, temperature=0)  # 글쓰기와 답변을 맡는 모델

# 판정만 맡는 Jev. 앞 단원과 같은 버전을 씁니다
jev = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
print("글쓰기:", model_name, "| 판정:", jev.model)

## 3. Jev로 번들 고르기

번들 고르기는 선택지 가운데 하나를 고르는 판정이라 Jev의 Choice가 맡습니다. 기준은 번들마다 무엇을 담는지이고, 그 설명은 번들 맨 위 목차에서 읽습니다. 위키와 관계없는 질문도 올 수 있으므로 `none`(어느 위키도 필요 없음) 선택지를 함께 둡니다.

In [ ]:
from langchain_typesafe import Choice


def bundle_summary(bundle):
    """번들 맨 위 목차(index.md)의 제목과 폴더 설명을 한 줄로 만듭니다. 이 한 줄이 그 번들을 고르는 기준입니다."""
    title = ""
    folders = []
    for line in (bundles_dir / bundle / "index.md").read_text(encoding="utf-8").splitlines():
        if line.startswith("# "):
            title = line[2:]
        if line.startswith("* ["):
            folders.append(line[3:].split("]")[0] + "(" + line.split(" - ")[-1] + ")")
    return title + ": " + ", ".join(folders)


# 번들 기준을 코드에 적지 않고 목차에서 읽으므로, 번들을 더하거나 바꿔도 이 코드는 그대로입니다
BUNDLE_CRITERIA = {}
for bundle in BUNDLES:
    BUNDLE_CRITERIA[bundle] = bundle_summary(bundle)

# 어느 위키도 읽을 필요가 없는 질문도 오므로, 그 선택지를 함께 둡니다
BUNDLE_CRITERIA["none"] = "어느 위키도 읽을 필요가 없는 질문(인사, 잡담, 회사 일과 관계없는 일반 질문)"

bundle_question = Choice(
    instructions="질문에 답하려면 어느 위키를 읽어야 하는지 고르세요.",
    criteria=BUNDLE_CRITERIA,
)


def route(question):
    """질문을 받아 Jev가 고른 번들 이름(위키가 필요 없으면 none)과 그 확률을 돌려줍니다."""
    decision = jev.invoke({"state": question, "questions": {"bundle": bundle_question}})
    answer = decision.choices["bundle"]
    return answer.choice, answer.probabilities[answer.choice]


BUNDLE_CRITERIA

In [ ]:
questions = [
    "7일 잔존율은 어떻게 계산하고, 최근 값은 얼마야?",
    "서버 장애가 나면 누구에게 먼저 연락해야 해?",
    "결제 전환율은 어떻게 계산하고, 최근 값은 얼마야?",
    "이번 주말에 볼 만한 영화 추천해 줘",
]

rows = []
for question in questions:
    bundle, probability = route(question)
    rows.append({"질문": question, "번들": bundle, "확률": round(probability, 2)})
route_table = pd.DataFrame(rows)
display(route_table)

## 4. 비서의 도구

### 4-1. 도구 두 개

- `read_index`: 목차 파일을 읽습니다. 비서는 늘 여기서 시작합니다. 없는 폴더를 물으면 오류 대신 다시 찾는 방법을 돌려줍니다(에이전트가 스스로 고칠 수 있게).
- `read_concept_doc`: 개념 문서 하나를 읽습니다. 상태, 신뢰 단계, 만료 여부는 코드가 계산해 문서 앞에 붙입니다. 모델이 frontmatter를 해석하다 틀리는 일을 줄입니다.

In [ ]:
import json

from langchain_core.tools import tool


@tool
def read_index(bundle: str, folder: str = "") -> str:
    """번들의 목차 파일(index.md)을 읽어 내용을 돌려줍니다.

    bundle: 번들 이름(예: studio).
    folder: 번들 안 폴더 이름. 비워 두면 번들 맨 위 목차를 읽습니다.
    예: bundle="studio"이면 studio/index.md, bundle="studio", folder="playbooks"이면 studio/playbooks/index.md
    """
    path = bundles_dir / bundle / folder / "index.md"

    # 없는 폴더를 물으면 에이전트가 다시 고를 수 있게 맨 위 목차부터 읽으라고 알려 줍니다
    if not path.exists():
        return f"{bundle}/{folder} 목차는 없습니다. 맨 위 목차(folder 비움)부터 읽으세요."
    return path.read_text(encoding="utf-8")


@tool
def read_concept_doc(bundle: str, concept_id: str) -> str:
    """개념 문서 하나를 읽어, 상태, 신뢰 단계, 만료 여부를 앞에 붙여 돌려줍니다.

    bundle: 번들 이름(예: studio).
    concept_id: 번들 안 경로에서 .md를 뺀 것. 폴더 목차에 적힌 파일 이름으로 만듭니다(예: projects/chorong_launch).
    """
    path = bundles_dir / bundle / f"{concept_id}.md"
    if not path.exists():
        return f"{bundle} 번들에 {concept_id} 문서는 없습니다. 목차에 적힌 파일 이름으로 개념 ID를 만드세요."
    meta, body = read_concept(path)

    # 답할 때 알려야 할 상태를 코드가 계산해 문서 앞에 붙입니다
    notes = [f"상태: {meta.get('status', 'stable')}", f"신뢰 단계: {trust_tier(meta)}"]
    if is_stale(meta, NOW):
        notes.append(f"만료: stale_after({meta['stale_after']})가 지났습니다")
    return "\n".join(notes) + "\n" + body

### 4-2. 도구를 직접 불러 보기

에이전트가 하는 순서대로 폴더 목차를 먼저 읽고, 그 목차에 적힌 문서를 엽니다.

In [ ]:
print(read_index.invoke({"bundle": "studio", "folder": "playbooks"}))

In [ ]:
print(read_concept_doc.invoke({"bundle": "studio", "concept_id": "playbooks/server_incident"}))

## 5. 비서 에이전트

### 5-1. 비서 만들기

비서의 규칙은 시스템 프롬프트에 적습니다. LLM 위키의 스키마 층에 해당합니다. 맨 위 목차와 오늘 날짜는 `ask`가 메시지에 넣습니다. 이 단계까지 모델에 맡기면 목차를 건너뛰고 폴더 이름을 짐작해 여는 일이 생기기 때문입니다. 반드시 해야 하는 단계는 코드가 하고, 무엇을 더 읽을지만 모델이 고릅니다. Jev가 `none`을 고른 질문은 비서를 부르지 않고 바로 답합니다. 그래도 폴더나 문서 이름을 짐작하는 일은 남을 수 있습니다. 그래서 규칙 1번에 개념 ID는 목차에 적힌 파일 이름으로 만들라고 적고, `read_index`는 없는 폴더를, `read_concept_doc`은 없는 문서를 물으면 다시 찾는 방법을 돌려줍니다.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage

SECRETARY_RULES = """너는 OKF 위키 비서다.
1. 메시지에 번들 맨 위 목차가 있다. 그것을 보고 필요한 폴더의 목차를 read_index로 읽은 뒤 문서를 연다. 개념 ID는 폴더 이름과 목차에 적힌 파일 이름에서 .md를 뺀 것이다(목차에 d7_retention.md가 있으면 metrics/d7_retention). 이름을 짐작해 만들지 않는다.
2. 답에 쓰는 사실은 read_concept_doc으로 읽은 문서에서만 가져온다. 문서에 없으면 모른다고 답한다.
3. read_concept_doc 결과 앞줄에 '만료' 줄이 있거나 상태가 draft, deprecated이면 답의 첫 줄을 '주의: '로 시작해 그 사실을 적는다.
4. 답은 한국어로 짧게 쓰고, 마지막 줄에 '출처: ' 뒤에 읽은 문서마다 '번들 이름/개념 ID(신뢰 단계)'를 쉼표로 이어 적는다. 예: 출처: game_data/metrics/d7_retention(사람 검토)"""

secretary = create_agent(
    llm,
    [read_index, read_concept_doc],
    system_prompt=SECRETARY_RULES,
)


def ask(question):
    """질문에 답하고 메시지 기록을 돌려줍니다.

    1. Jev로 번들을 고릅니다.
    2. 위키가 필요 없는 질문(none)이면 비서를 부르지 않고, 위키를 읽지 않았다고 답합니다.
    3. 번들이 정해지면 그 번들의 맨 위 목차와 오늘 날짜를 붙여 비서에게 묻습니다.
    """
    bundle, probability = route(question)

    # 위키가 필요 없는 질문이면 비서(에이전트)를 부르지 않습니다
    if bundle == "none":
        return [AIMessage("이 질문은 위키에서 찾을 내용이 아니라 위키를 읽지 않았습니다.\n출처: 없음")]

    # 목차부터 읽는 단계는 모델에 맡기지 않고 코드가 넣어 줍니다
    root_index = (bundles_dir / bundle / "index.md").read_text(encoding="utf-8")
    content = f"오늘: {NOW.date().isoformat()}\n번들: {bundle}\n번들 맨 위 목차:\n{root_index}\n질문: {question}"
    result = secretary.invoke({"messages": [{"role": "user", "content": content}]})
    return result["messages"]


def show_trace(messages):
    """도구 호출 순서와 마지막 답을 출력합니다."""
    step = 0
    for message in messages:
        if message.type != "ai":
            continue
        for call in message.tool_calls:
            step = step + 1
            print(f"[{step}] {call['name']} {json.dumps(call['args'], ensure_ascii=False)}")
    print("\n답:\n" + messages[-1].text)

### 5-2. 데이터 위키: 지표 정의 찾기

In [ ]:
question = questions[0]
print("질문:", question)
show_trace(ask(question))

### 5-3. 업무 위키: 만료된 문서로 답하기

In [ ]:
question = questions[1]
print("질문:", question)
show_trace(ask(question))

### 5-4. 데이터 위키: 초안 문서로 답하기

결제 전환율 문서는 초안이고 계산 방법과 최근 값이 아직 없습니다. 비서가 초안이라고 알리고, 문서에 없는 값은 지어내지 않고 모른다고 답하는지 봅니다. 5-2의 7일 잔존율(사람 검토)과 답이 어떻게 다른지 견줘 봅니다.

In [ ]:
question = questions[2]
print("질문:", question)
show_trace(ask(question))

### 5-5. 위키가 필요 없는 질문

Jev가 `none`을 고르면 비서는 도구를 하나도 부르지 않고 바로 답합니다. 모델을 부르지 않으니 비용도 들지 않습니다.

In [ ]:
question = questions[3]
print("질문:", question)
show_trace(ask(question))

## 6. 핵심 정리

- **번들 고르기:** 선택지 하나를 고르는 판정이라 Jev Choice가 맡고, 글쓰기와 답변은 GPT가 맡습니다. 위키가 필요 없는 질문은 `none`으로 걸러 비서를 부르지 않습니다.
- **목차부터:** 비서는 `index.md`부터 읽고 필요한 문서만 엽니다. 임베딩 검색 없이도 수십 개 문서 규모에서는 충분합니다.
- **일반화:** 번들 고르기 기준은 목차에서, 문서 상태는 frontmatter에서 코드가 읽습니다. 위키를 바꿔도 비서 코드는 그대로입니다.
- **신뢰 표시:** 상태, 신뢰 단계, 만료는 코드가 계산해 문서에 붙이고, 비서는 그것을 답에서 알립니다.